# Model Training 

- Train with tennis eras and test with 2024 matches
    
    - 1968-2023 (Open era)
    - 1990-2023 (Modern era)
    - 2024 (Tests Only)

- Use XGBoost model to predict


In [74]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import KFold, cross_val_score
from joblib import dump

## Load Data

In [75]:
target = pd.read_csv('../data/processed/5.1_data_target.csv', sep=';', encoding='utf-8')
target.head(3)

,SCORE
0,0
1,1
2,0


In [76]:
feature = pd.read_csv('../data/processed/5.1_data_feature.csv', sep=';', encoding='utf-8')
feature['SCORE'] = target['SCORE']
feature.head(3)

,TOURNEY_DATE,BEST_OF,AGE_DIFF,HT_DIFF,0_WIN_VS_1,1_WIN_VS_0,TOTAL_MATCHES,0_WIN_RATE,1_WIN_RATE,WIN_RATE_DIFF,...,PERC_1_WIN_LAST_25,PERC_1_WIN_LAST_50,PERC_1_WIN_LAST_100,PERC_1_WIN_COMBINED,LAST_10_WIN_DIFF,LAST_25_WIN_DIFF,LAST_50_WIN_DIFF,LAST_100_WIN_DIFF,WIN_COMBINED_DIFF,SCORE
0,1967-12-28,5,-5.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0
1,1967-12-28,3,2.5,-1.0,0.0,0.0,0.0,0.0,0.0,-0.0,...,0.04,0.02,0.01,0.06,-0.1,-0.04,-0.02,-0.01,-0.06,1
2,1967-12-28,3,7.4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0


In [77]:
feature_robust = pd.read_csv('../data/processed/5.2_data_feature_robust_esc.csv', sep=';', encoding='utf-8')
feature_robust['SCORE'] = target['SCORE']
feature_robust.head(3)

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,TOURNEY_DATE,SCORE
0,2.0,-0.763889,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.747788,...,-1.714286,-1.095238,-1.678571,0.00,0.000,0.000000,0.000000,0.000000,1967-12-28,0
1,0.0,0.347222,-0.1,0.0,0.0,0.0,0.0,0.0,-0.0,-1.747788,...,-1.642857,-1.071429,-1.464286,-0.25,-0.125,-0.055556,-0.026316,-0.166667,1967-12-28,1
2,0.0,1.027778,0.1,0.0,0.0,0.0,0.0,0.0,0.0,-1.780659,...,-1.714286,-1.095238,-1.678571,0.00,0.000,0.000000,0.000000,0.000000,1967-12-28,0


In [78]:
feature_std = pd.read_csv('../data/processed/5.2_data_feature_std_esc.csv', sep=';', encoding='utf-8')
feature_std['SCORE'] = target['SCORE']
feature_std.head(3)

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,TOURNEY_DATE,SCORE
0,1.888487,-0.994933,-0.002064,-0.418707,-0.416525,-0.49076,-0.578864,-0.578392,-0.000583,-2.036470,...,-1.973571,-1.664340,-2.167572,-0.001103,-0.000711,-0.003203,-0.002652,-0.001656,1967-12-28,0
1,-0.528503,0.454129,-0.118093,-0.418707,-0.416525,-0.49076,-0.578864,-0.578392,-0.000583,-2.036470,...,-1.884794,-1.623651,-1.884118,-0.339369,-0.149579,-0.075972,-0.036112,-0.228564,1967-12-28,1
2,-0.528503,1.341679,0.113964,-0.418707,-0.416525,-0.49076,-0.578864,-0.578392,-0.000583,-2.077725,...,-1.973571,-1.664340,-2.167572,-0.001103,-0.000711,-0.003203,-0.002652,-0.001656,1967-12-28,0


In [79]:
feature_pca = pd.read_csv('../data/processed/5.3_data_feature_robust_pca.csv', sep=';', encoding='utf-8')
feature_pca['SCORE'] = target['SCORE']
feature_pca.head(3)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,TOURNEY_DATE,SCORE
0,-3.951160,-0.111719,4.322574,0.068662,0.107928,1.112041,0.616423,0.719888,-0.695141,-0.028404,0.608169,-0.028454,0.049258,0.002598,-0.093784,1967-12-28,0
1,-3.829423,-0.358895,4.048304,0.096652,-0.126584,-0.731900,-0.111335,0.373972,0.336236,0.019632,0.159203,-0.098360,0.118925,0.026080,0.028853,1967-12-28,1
2,-3.943885,0.026808,4.274866,-0.016315,-0.031845,-0.778547,0.073513,0.308310,1.032122,0.049826,0.287799,-0.001905,0.021293,-0.007737,0.106232,1967-12-28,0


In [80]:
feature_lda = pd.read_csv('../data/processed/5.3_data_feature_robust_lda.csv', sep=';', encoding='utf-8')
feature_lda['SCORE'] = target['SCORE']
feature_lda.head(3)

,0,TOURNEY_DATE,SCORE
0,-0.283179,1967-12-28,0
1,0.253717,1967-12-28,1
2,0.435131,1967-12-28,0


## Train and Test Data

#### 1968-2024 (Train - Open era)

In [94]:
# open_data = feature_robust
# open_data = feature_std
# open_data = feature_pca
# open_data = feature_lda
open_data = feature
open_data.tail(3)

,TOURNEY_DATE,BEST_OF,AGE_DIFF,HT_DIFF,0_WIN_VS_1,1_WIN_VS_0,TOTAL_MATCHES,0_WIN_RATE,1_WIN_RATE,WIN_RATE_DIFF,...,PERC_1_WIN_LAST_25,PERC_1_WIN_LAST_50,PERC_1_WIN_LAST_100,PERC_1_WIN_COMBINED,LAST_10_WIN_DIFF,LAST_25_WIN_DIFF,LAST_50_WIN_DIFF,LAST_100_WIN_DIFF,WIN_COMBINED_DIFF,SCORE
187730,2024-12-18,5,-0.7,5.0,1.0,0.0,1.0,1.0,0.0,1.0,...,0.20,0.10,0.05,0.29,0.0,0.24,0.12,0.06,0.10,0
187731,2024-12-18,5,0.2,13.0,0.0,0.0,0.0,0.0,0.0,-0.0,...,0.20,0.10,0.05,0.29,0.2,0.44,0.46,0.23,0.32,1
187732,2024-12-18,5,1.3,-8.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.64,0.56,0.28,0.61,-0.1,-0.12,-0.02,0.27,-0.05,0


In [95]:
open_target = open_data['SCORE']
open_feature = open_data.drop(['SCORE', 'TOURNEY_DATE'], axis=1)

In [96]:
X_train_open, X_test_open, y_train_open, y_test_open = train_test_split(open_feature, open_target, train_size=0.7, random_state=0)

#### 1990-2024 (Train - Modern era)

In [101]:
# modern_data = feature_robust[feature_robust['TOURNEY_DATE'] >= '1990-01-01']
# modern_data = feature_std[feature_std['TOURNEY_DATE'] >= '1990-01-01']
# modern_data = feature_pca[feature_pca['TOURNEY_DATE'] >= '1990-01-01']
# modern_data = feature_lda[feature_lda['TOURNEY_DATE'] >= '1990-01-01']
# modern_data = feature_partial_pca[feature_partial_pca['TOURNEY_DATE'] >= '1990-01-01']
modern_data = feature[feature['TOURNEY_DATE'] >= '1990-01-01']
modern_data.head(3)

,TOURNEY_DATE,BEST_OF,AGE_DIFF,HT_DIFF,0_WIN_VS_1,1_WIN_VS_0,TOTAL_MATCHES,0_WIN_RATE,1_WIN_RATE,WIN_RATE_DIFF,...,PERC_1_WIN_LAST_25,PERC_1_WIN_LAST_50,PERC_1_WIN_LAST_100,PERC_1_WIN_COMBINED,LAST_10_WIN_DIFF,LAST_25_WIN_DIFF,LAST_50_WIN_DIFF,LAST_100_WIN_DIFF,WIN_COMBINED_DIFF,SCORE
79194,1990-01-01,3,2.9,-26.0,0.0,0.0,0.0,0.0,0.0,-0.0,...,0.32,0.20,0.10,0.35,-0.4,-0.00,0.04,0.02,-0.15,1
79195,1990-01-01,3,0.1,12.0,0.0,0.0,0.0,0.0,0.0,-0.0,...,0.48,0.38,0.25,0.45,-0.1,-0.00,0.18,0.27,0.02,1
79196,1990-01-01,3,0.4,-13.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.12,0.06,0.03,0.17,0.2,0.32,0.16,0.08,0.22,0


In [102]:
modern_target = modern_data['SCORE']
modern_feature = modern_data.drop(['SCORE', 'TOURNEY_DATE'], axis=1)

In [103]:
X_train_modern, X_test_modern, y_train_modern, y_test_modern = train_test_split(modern_feature, modern_target, train_size=0.7, random_state=0)

## XGBoost

#### Open

In [104]:
xgb_open = XGBClassifier(max_depth=3,
    learning_rate=0.1,
    n_estimators=500,
    subsample=0.6,
    colsample_bytree=0.6,
    gamma=2,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    objective='binary:logistic'
    )

In [105]:
xgb_open.fit(X_train_open, y_train_open)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.6
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [106]:
import pandas as pd

importances = xgb_open.feature_importances_
features = X_train_open.columns

df_importance = pd.DataFrame({'feature': features, 'importance': importances})
df_importance = df_importance.sort_values('importance', ascending=False)
print(df_importance)

                    feature  importance
11              RATING_DIFF    0.217565
14      SURFACE_RATING_DIFF    0.147734
28        LAST_100_WIN_DIFF    0.119043
27         LAST_50_WIN_DIFF    0.056169
2                   HT_DIFF    0.046955
1                  AGE_DIFF    0.044749
23      PERC_1_WIN_LAST_100    0.028126
29        WIN_COMBINED_DIFF    0.026553
10          PLAYER_1_RATING    0.025832
18      PERC_0_WIN_LAST_100    0.023467
9           PLAYER_0_RATING    0.023392
0                   BEST_OF    0.021500
12  PLAYER_0_SURFACE_RATING    0.020137
13  PLAYER_1_SURFACE_RATING    0.017023
6                0_WIN_RATE    0.016137
16       PERC_0_WIN_LAST_25    0.016107
8             WIN_RATE_DIFF    0.015103
17       PERC_0_WIN_LAST_50    0.014707
19      PERC_0_WIN_COMBINED    0.013180
7                1_WIN_RATE    0.012277
26         LAST_25_WIN_DIFF    0.011929
22       PERC_1_WIN_LAST_50    0.011623
24      PERC_1_WIN_COMBINED    0.011316
21       PERC_1_WIN_LAST_25    0.010662


In [107]:
predict_xgb_open = xgb_open.predict(X_test_open)
predict_xgb_open

array([0, 1, 0, ..., 0, 1, 0], shape=(56320,))

In [108]:
accuracy_xgb_open = accuracy_score(y_test_open, predict_xgb_open)
print(f'Accuracy: {(accuracy_xgb_open * 100):.2f}%')

Accuracy: 71.27%


In [109]:
confusion_matrix(y_test_open, predict_xgb_open)

array([[19903,  8217],
       [ 7966, 20234]])

In [110]:
print(classification_report(y_test_open, predict_xgb_open))

              precision    recall  f1-score   support

           0       0.71      0.71      0.71     28120
           1       0.71      0.72      0.71     28200

    accuracy                           0.71     56320
   macro avg       0.71      0.71      0.71     56320
weighted avg       0.71      0.71      0.71     56320



In [111]:
kfold = KFold(n_splits=5, shuffle=True, random_state=5)

In [112]:
xgb_open2 = XGBClassifier(max_depth=3,
    learning_rate=0.1,
    n_estimators=500,
    subsample=0.6,
    colsample_bytree=0.6,
    gamma=2,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    objective='binary:logistic')
result = cross_val_score(xgb_open2, open_feature, open_target, cv=kfold)
print(f'Accuracy: {(result.mean() * 100):.2f} %')

Accuracy: 71.36 %


#### Modern

In [61]:
xgb_modern = XGBClassifier(max_depth=3,
    learning_rate=0.1,
    n_estimators=500,
    subsample=0.6,
    colsample_bytree=0.6,
    gamma=2,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    objective='binary:logistic')

In [62]:
xgb_modern.fit(X_train_modern, y_train_modern)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.6
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [63]:
predict_xgb_modern = xgb_modern.predict(X_test_modern)
predict_xgb_modern

array([0, 0, 0, ..., 0, 0, 0], shape=(32562,))

In [64]:
accuracy_xgb_modern = accuracy_score(y_test_modern, predict_xgb_modern)
print(f'Accuracy: {(accuracy_xgb_modern * 100):.2f}%')

Accuracy: 68.55%


In [65]:
confusion_matrix(y_test_modern, predict_xgb_modern)

array([[11088,  5069],
       [ 5173, 11232]])

In [66]:
print(classification_report(y_test_modern, predict_xgb_modern))

              precision    recall  f1-score   support

           0       0.68      0.69      0.68     16157
           1       0.69      0.68      0.69     16405

    accuracy                           0.69     32562
   macro avg       0.69      0.69      0.69     32562
weighted avg       0.69      0.69      0.69     32562



In [67]:
kfold2 = KFold(n_splits=5, shuffle=True, random_state=5)

In [68]:
xgb_modern2 = XGBClassifier(max_depth=3,
    learning_rate=0.1,
    n_estimators=500,
    subsample=0.6,
    colsample_bytree=0.6,
    gamma=2,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    objective='binary:logistic')
result2 = cross_val_score(xgb_modern2, modern_feature, modern_target, cv=kfold2)
print(f'Accuracy: {(result2.mean() * 100):.2f}%')

Accuracy: 68.74%


In [113]:
#save model
import joblib

final_xgb = XGBClassifier(max_depth=3,
    learning_rate=0.1,
    n_estimators=500,
    subsample=0.6,
    colsample_bytree=0.6,
    gamma=2,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    objective='binary:logistic'
    )

final_xgb.fit(open_feature, open_target)

joblib.dump(final_xgb, '../models/xgboost_model.joblib')

['../models/xgboost_model.joblib']

# Model Training Result

Best dataset
- XGBoost using raw open data
    - Accuracy: 71.27%
    - Confusion Matrix 
        - array([[19903,  8217],
       [ 7966, 20234]])
    - Cross Validation 71.36%

TODO:
- Test with recent games (2025)